# 311 cleaning and coverage

Run the CLI command in README first. This notebook reads the same hash-verified outputs. It fetches no data and does not depend on a previous notebook kernel. Counts refer to accepted service requests, not confirmed complaints caused by visitors. Unknown request categories remain in the denominator. Zero counts require explicitly declared complete source coverage.

In [ ]:
from pathlib import Path
import json
import os
import pandas as pd
from tourism_pipeline import sha256
run_dir = Path(os.environ.get("TOURISM_RUN", "runs/fixture"))
assert json.loads((run_dir / "status.json").read_text())["state"] == "complete"
provenance = json.loads((run_dir / "provenance.json").read_text())
for name, expected in provenance["output_sha256"].items():
    assert sha256(run_dir / name) == expected, name
print("SYNTHETIC SOFTWARE FIXTURE" if provenance["config"]["is_fixture"] else "Observed snapshots; descriptive associations only")


In [ ]:
audit = json.loads((run_dir / "audit.json").read_text())
assert audit["raw_request_rows"] == audit["accepted_unique_requests"] + audit["duplicate_rows_removed"] + audit["rejected_rows"]
audit

In [ ]:
monthly = pd.read_csv(run_dir / "monthly_311.csv")
assert not monthly.duplicated(["city", "month"]).any()
monthly.groupby("city").agg(months=("month", "size"), declared_complete=("request_coverage_complete", "sum"), accepted_requests=("n_requests", "sum"))

In [ ]:
pd.read_csv(run_dir / "rejected_requests.csv").groupby(["city", "reason"]).size()